In [ ]:
# Import necessary libraries
import os
import glob
import pandas as pd
import numpy as np
from datetime import datetime, timedelta
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import gc

In [ ]:
# Define friendly name mapping (from multi_plot_add_acoustic.ipynb)
pv_mapping = [
    # --- Vacuum Pressure Sensors ---

    # --- Radiation and Detectors ---
    ("B2_VXD:Rad:Res:BPATDCU3:A1:DoseRate", "B_DMD_DoseRate"),
    ("B2_VXD:Rad:QCS_FW_225:DoseRate", "B_DMD_FW_225L"),
    ("B2_VXD:Rad:QCS_FW_135:DoseRate", "B_DMD_FW_135L"),
    ("B2_CDC:CUR:LOGGER:L15_2:MAX", "B_CDC_Imax_uA"),
    ("B2_CDC:CUR_AVERAGE", "B_CDC_Iavg_uA"),
    ("B2_TOP:TYPE2_avg_scalerRate_PMT17-32", "B_PMThits_Hz"),
    ("B2_TOP:TYPE2_avg_scalerRate_PMT17-32_60", "B_PMThits_avg60s"),
    ("B2_nsm:get:TRGOSCILLO0:lff_ler", "B_ECL_BG_duration_ms"),
    ("B2_nsm:get:ECLTRG_FAM:rate_bw", "B_ECL_BW"),

    # --- Beam Parameters and Injection ---
    ("B2_nsm:get:ECL_LUM_MON:lum_acc_20", "A_LUMI_30"),
    ("CG_OPR:SpecificLuminosity", "A_LUMI_SP_30"),
    ("BMLDCCT:CURRENT", "A_LER_Current_mA"),
    ("BMLDCCT:RATE", "A_LER_Inj_Rate_mAps"),
    ("BMLDCCT:LIFE", "A_LER_Lifetime_min"),
    ("CGLOPT:IP:BETA_Y", "A_LER_BetaY_IP_m"),
    ("CGLOPT:IP:BETA_X", "A_LER_BetaX_IP_m"),
    ("BMLXRM:BEAM:SIGMAX", "A_LER_SigmaX_IP_um"),
    ("BMLXRM:BEAM:SIGMAY", "A_LER_SigmaY_IP_um"),
    ("CG_OPT:CAP:SIGMAY", "A_LER_CapSigmaY_um"),
    ("BTpBPM:QMD11P_K_1:NC_1Hz:C", "A_Q_LER_BT_end_nC"),
    ("LIiBM:SP_61_8_1:ISNGL:KBP", "A_Q_LER_Linac_end_nC"),
    ("CGLINJ:EFFICIENCY", "A_LER_Tot_INJ_Effi"),
    ("LIiEV:BEAM_REP:READ:KBP", "A_LER_Rep_ep_Hz"),
    ("CGLINJ:KICKER:HEIGHT_R", "A_LER_Kicker_Height_mm"),
    ("CGLINJ:KICKER:JUMP_R", "A_LER_Kicker_Jump"),
    ("CGLINJ:SEPTUM:POS_R", "A_LER_Septum_Pos_mm"),
    ("CGLINJ:SEPTUM:ANG_R", "A_LER_Septum_Ang_mm"),
    ("LIiRF:MOPS:SET_PHASE:LER", "A_LER_INJ_Phase"),
    ("CGLINJ:INJECTION:YPOS", "A_LER_INJ_PosY_m"),
    ("CGLINJ:INJECTION:YANG", "A_LER_INJ_AngY_rad"),
    ("BMLD07:INJ:X", "A_LER_INJ_D7_BPMX_mm"),
    ("BMLD07:INJ:Y", "A_LER_INJ_D7_BPMY_mm"),
    ("BMLD07:INJ:Q", "A_LER_INJ_D7_Charge"),
    ("VALCLM:D06CV1TOP:RQ:SET_POS", "A_D6V1_TOP_Head_Pos_Set"),
    ("VALCLM:D06CV1TOP:ST:POS", "A_D6V1_TOP_Head_Pos_Meas"),
    ("VALCLM:D06CV1BTM:RQ:SET_POS", "A_D6V1_BTM_Head_Pos_Set"),
    ("VALCLM:D06CV1BTM:ST:POS", "A_D6V1_BTM_Head_Pos_Meas"),
    ("BML:MQTAFOP1:POS.PXP", "A_D6V1_Upstrm_BPM_PosX"),
    ("BML:MQTAFOP1:POS.PYP", "A_D6V1_Upstrm_BPM_PosY"),
    ("BML:MQT3FOP1:POS.PXP", "A_D6V1_Downstrm_BPM_PosX"),
    ("BML:MQT3FOP1:POS.PYP", "A_D6V1_Downstrm_BPM_PosY"),
    ("BM_BLM:BTCBT:ADC:MEAN", "A_INJ_Loss_Monitor"),

    # --- Acoustic Sensors ---
    ("RFLAE:D06:OSC1:CH1:MIN", "ACOU_Top_Min"),
    ("RFLAE:D06:OSC1:CH1:MAX", "ACOU_Top_Max"),
    ("RFLAE:D06:OSC1:CH1:VPP", "ACOU_Top_Vpp"),
    ("RFLAE:D06:OSC1:CH2:MIN", "ACOU_BmLoss_Min"),
    ("RFLAE:D06:OSC1:CH2:MAX", "ACOU_BmLoss_Max"),
    ("RFLAE:D06:OSC1:CH2:VPP", "ACOU_BmLoss_Vpp"),
    ("RFLAE:D06:OSC1:CH3:MIN", "ACOU_Btm_Min"),
    ("RFLAE:D06:OSC1:CH3:MAX", "ACOU_Btm_Max"),
    ("RFLAE:D06:OSC1:CH3:VPP", "ACOU_Btm_Vpp")
    
    # --- Add HER ---
]

# Create mapping dictionary for fast lookup
pv_dict = {k: v for k, v in pv_mapping}

# Function to get friendly name or return original column name if not found
def get_friendly_name(col_name):
    return pv_dict.get(col_name, col_name)

print(f"Loaded {len(pv_mapping)} PV name mappings")

# Load machine data from Machine_Inj_BG_byDay folder
# Extract MMDD from date_str (e.g., "20260223" -> "0223")
mmdd = date_str[4:8]  # Get MMDD from YYYYMMDD
machine_base_path = "/Users/xylu/Desktop/Data/Machine_Inj_BG_byDay"
machine_folder = os.path.join(machine_base_path, mmdd)

if not os.path.exists(machine_folder):
    print(f"⚠ Machine folder not found: {machine_folder}")
    machine_df = None
else:
    print(f"✓ Found Machine folder: {machine_folder}")
    
    # Find CSV files in Machine folder
    machine_csv_files = sorted(glob.glob(os.path.join(machine_folder, "*.csv")))
    
    if len(machine_csv_files) == 0:
        print(f"  ⚠ No CSV files found in Machine folder")
        machine_df = None
    else:
        print(f"  Found {len(machine_csv_files)} machine data file(s)")
        
        # Load and combine all machine data files
        machine_data_list = []
        
        for csv_file in machine_csv_files:
            try:
                df_temp = pd.read_csv(csv_file)
                # Convert timestamp column to datetime (handle both #date and Timestamp)
                if '#date' in df_temp.columns:
                    df_temp['#date'] = pd.to_datetime(df_temp['#date'], format='ISO8601')
                    df_temp.rename(columns={'#date': 'time_datetime'}, inplace=True)
                elif 'Timestamp' in df_temp.columns:
                    df_temp['Timestamp'] = pd.to_datetime(df_temp['Timestamp'], format='ISO8601')
                    df_temp.rename(columns={'Timestamp': 'time_datetime'}, inplace=True)
                machine_data_list.append(df_temp)
                print(f"  ✓ {os.path.basename(csv_file)}: {len(df_temp)} samples, {len(df_temp.columns)} columns")
            except Exception as e:
                print(f"  ✗ Error reading {os.path.basename(csv_file)}: {e}")
        
        if machine_data_list:
            # Combine all machine data
            machine_df = pd.concat(machine_data_list, ignore_index=True)
            machine_df = machine_df.sort_values('time_datetime').reset_index(drop=True)
            
            # Rename columns to friendly names
            machine_df.rename(columns=get_friendly_name, inplace=True)
            
            print(f"\n✓ Combined machine data: {len(machine_df)} samples")
            print(f"  Time range: {machine_df['time_datetime'].min()} to {machine_df['time_datetime'].max()}")
            print(f"  Columns ({len(machine_df.columns)}): {list(machine_df.columns)}")
            
            # Filter by time range
            mask = (machine_df['time_datetime'] >= start_time_obj) & (machine_df['time_datetime'] <= end_time_obj)
            machine_df_filtered = machine_df[mask].reset_index(drop=True)
            
            print(f"\n✓ Filtered machine data: {len(machine_df_filtered)} samples")
            
            # ===== CREATE FILTERED ARRAYS IMMEDIATELY AFTER DATA IMPORT =====
            # Only use positive injection rate and non-zero efficiency for all plots
            
            # Get positive injection rate (>= 0)
            if 'A_BM_Inj_Rate_mAps' in machine_df_filtered.columns:
                mask_inj_positive = (machine_df_filtered['A_BM_Inj_Rate_mAps'].notna()) & (machine_df_filtered['A_BM_Inj_Rate_mAps'] >= 0)
                inj_rate_positive = machine_df_filtered[mask_inj_positive].copy()
                inj_rate_positive_times = inj_rate_positive['time_datetime']
                inj_rate_positive_values = inj_rate_positive['A_BM_Inj_Rate_mAps'].values
            else:
                inj_rate_positive_times = pd.Series([], dtype='datetime64[ns]')
                inj_rate_positive_values = np.array([])
            
            # Get non-zero efficiency (!= 0)
            if 'A_INJ_Effi' in machine_df_filtered.columns:
                mask_effi_nonzero = (machine_df_filtered['A_INJ_Effi'].notna()) & (machine_df_filtered['A_INJ_Effi'] != 0)
                effi_nonzero = machine_df_filtered[mask_effi_nonzero].copy()
                effi_nonzero_times = effi_nonzero['time_datetime']
                effi_nonzero_values = effi_nonzero['A_INJ_Effi'].values
            else:
                effi_nonzero_times = pd.Series([], dtype='datetime64[ns]')
                effi_nonzero_values = np.array([])
            
            print(f"  ✓ Injection Rate (positive): {len(inj_rate_positive_values)} pts")
            print(f"  ✓ Efficiency (non-zero): {len(effi_nonzero_values)} pts")
        else:
            print("⚠ No machine data loaded!")
            machine_df = None
            machine_df_filtered = None
            inj_rate_positive_times = pd.Series([], dtype='datetime64[ns]')
            inj_rate_positive_values = np.array([])
            effi_nonzero_times = pd.Series([], dtype='datetime64[ns]')
            effi_nonzero_values = np.array([])

In [ ]:
# Load abort event data
import subprocess
import sys

# Ensure openpyxl is installed
subprocess.check_call([sys.executable, "-m", "pip", "install", "openpyxl", "-q"])

abort_data_path = "/Users/xylu/Desktop/Data/Complete_LER_Event_Data_Analysis_Summary.xlsx"

print("="*60)
print("Loading Abort Event Data")
print("="*60)

if os.path.exists(abort_data_path):
    # Load abort data from Excel file
    abort_df = pd.read_excel(abort_data_path)
    
    # Parse the Time column (format: "M/D/YY H:MM" or "M/D/YY HH:MM")
    # Convert to datetime
    abort_df['abort_datetime'] = pd.to_datetime(abort_df['Time'], format='%m/%d/%y %H:%M')
    
    print(f"✓ Loaded {len(abort_df)} abort events")
    print(f"  Date range: {abort_df['abort_datetime'].min()} to {abort_df['abort_datetime'].max()}")
    print(f"  Columns: {list(abort_df.columns)}")
    
    # Filter abort events for the current date
    abort_date_start = datetime.strptime(date_str, "%Y%m%d").replace(hour=0, minute=0, second=0)
    abort_date_end = abort_date_start + timedelta(days=1)
    
    abort_mask = (abort_df['abort_datetime'] >= abort_date_start) & (abort_df['abort_datetime'] < abort_date_end)
    abort_today = abort_df[abort_mask].copy()
    
    print(f"\n✓ Found {len(abort_today)} abort event(s) on {date_str}")
    
    if len(abort_today) > 0:
        print("\nAbort Events:")
        for idx, row in abort_today.iterrows():
            print(f"  {row['abort_datetime'].strftime('%Y-%m-%d %H:%M:%S')} - {row['Category']} - {row['Origin']}")
    else:
        print("  No abort events found for this date")
else:
    print(f"⚠ Abort data file not found: {abort_data_path}")
    abort_df = None
    abort_today = None
    

In [ ]:
# Plot machine variables in separate dual-axis figures
# Figure 1: Vpp + Beam Current
# Figure 2: Injection Rate + Injection Efficiency
# Figure 3: Charge + Injection Rep Rate
# Figure 4: Total Loss (Effi), Life Loss, Inj. Loss (Effi)
# Figure 5: Beam Lifetime + Beam Current
# Figure 6: Total Inj (Q), Life Loss, Inj Rate, Inj. Loss (Q)
# Figure 7: Imported Efficiency + Calculated Efficiency
# Figure 8: Inj. Loss (Effi) vs Inj. Loss (Q)
if filtered_df is not None and len(filtered_df) > 0 and machine_df_filtered is not None and len(machine_df_filtered) > 0:

    print(f"\n{'='*70}")
    print("Creating 8-panel daily plot")
    print(f"{'='*70}")

    # Create figure with 8 subplots in a row
    fig, (ax1, ax2, ax3, ax4, ax5, ax6, ax7, ax8) = plt.subplots(8, 1, figsize=(12, 22))

    # Add figure title with date
    date_formatted = datetime.strptime(date_str, "%Y%m%d").strftime("%Y-%m-%d")
    fig.suptitle(f'Acoustic, current, injection - {date_formatted}', fontsize=12, fontweight='bold', y=0.995)

    # ===== FIGURE 1: Vpp + Beam Current =====
    if 'A_BM_Current_mA' in machine_df_filtered.columns:
        # Get data
        mask_current = machine_df_filtered['A_BM_Current_mA'].notna()
        valid_times_current = machine_df_filtered.loc[mask_current, 'time_datetime']
        valid_current = machine_df_filtered.loc[mask_current, 'A_BM_Current_mA']

        # Plot Vpp on left y-axis
        color_vpp = 'tab:blue'
        ax1.set_ylabel('Bottom Acoustic Vpp (V)', color=color_vpp, fontsize=14, fontweight='bold')
        ax1.plot(filtered_df['time_datetime'], filtered_df['vpp_volts'],
                linewidth=2.5, color=color_vpp, alpha=0.7, label=f'CH3 Vpp ({len(filtered_df)} pts)')
        ax1.tick_params(axis='y', labelcolor=color_vpp, labelsize=12)
        ax1.set_ylim(0, 18)
        ax1.grid(True, alpha=0.3)

        # Plot Beam Current on right y-axis
        ax1_right = ax1.twinx()
        color_current = 'darkorange'
        ax1_right.set_ylabel('Beam Current (mA)', color=color_current, fontsize=14, fontweight='bold')
        ax1_right.plot(valid_times_current, valid_current,
                      linewidth=2.5, color=color_current, alpha=0.7, label=f'Beam Current ({len(valid_current)} pts)')
        ax1_right.tick_params(axis='y', labelcolor=color_current, labelsize=12)
        ax1_right.set_ylim(0, 1700)

        print(f"  ✓ Figure 1: Vpp ({len(filtered_df)} pts) + Beam Current ({len(valid_current)} pts)")

    # ===== FIGURE 2: Injection Rate + Injection Efficiency =====
    # Use pre-filtered arrays created immediately after data import
    if len(inj_rate_positive_values) > 0 and len(effi_nonzero_values) > 0:
        # Plot Injection Rate on left y-axis
        color_inj_rate = 'tab:green'
        ax2.set_ylabel('Injection Rate (mA/s)', color=color_inj_rate, fontsize=14, fontweight='bold')
        ax2.plot(inj_rate_positive_times, inj_rate_positive_values,
                linewidth=0, marker='o', markersize = 3, color=color_inj_rate, alpha=0.7, label=f'Inj Rate (positive: {len(inj_rate_positive_values)} pts)')
        ax2.tick_params(axis='y', labelcolor=color_inj_rate, labelsize=12)
        ax2.set_ylim(0, 6)
        ax2.grid(True, alpha=0.3)

        # Plot Injection Efficiency on right y-axis
        ax2_right = ax2.twinx()
        color_effi = 'tab:red'
        ax2_right.set_ylabel('Injection efficiency (%)', color=color_effi, fontsize=14, fontweight='bold')
        ax2_right.plot(effi_nonzero_times, effi_nonzero_values,
                      linewidth=0, marker='s', markersize = 3,color=color_effi, alpha=0.7, label=f'Efficiency (non-zero: {len(effi_nonzero_values)} pts)')
        ax2_right.tick_params(axis='y', labelcolor=color_effi, labelsize=12)
        ax2_right.set_ylim(0, 110)

        print(f"  ✓ Figure 2: Inj Rate (positive: {len(inj_rate_positive_values)} pts) + Efficiency (non-zero: {len(effi_nonzero_values)} pts)")

    # ===== FIGURE 3: Charge + Injection Rep Rate =====
    if 'A_Qep_BT_end_nC' in machine_df_filtered.columns and 'A_INJ_Rep_ep_Hz' in machine_df_filtered.columns:
        # Get Charge
        mask_charge = machine_df_filtered['A_Qep_BT_end_nC'].notna()
        valid_times_charge = machine_df_filtered.loc[mask_charge, 'time_datetime']
        valid_charge = machine_df_filtered.loc[mask_charge, 'A_Qep_BT_end_nC']

        # Get Rep Rate
        mask_rep = machine_df_filtered['A_INJ_Rep_ep_Hz'].notna()
        valid_times_rep = machine_df_filtered.loc[mask_rep, 'time_datetime']
        valid_rep = machine_df_filtered.loc[mask_rep, 'A_INJ_Rep_ep_Hz']

        # Plot Charge on left y-axis
        color_charge = 'tab:brown'
        ax3.set_ylabel('Charge BT end (nC)', color=color_charge, fontsize=14, fontweight='bold')
        ax3.plot(valid_times_charge, valid_charge,
                linewidth=0, marker='o', color=color_charge, alpha=0.7, label=f'Charge ({len(valid_charge)} pts)')
        ax3.tick_params(axis='y', labelcolor=color_charge, labelsize=12)
        ax3.set_ylim(0, 3)
        ax3.grid(True, alpha=0.3)

        # Plot Injection Rep Rate on right y-axis
        ax3_right = ax3.twinx()
        color_rep = 'tab:purple'
        ax3_right.set_ylabel('Injection Rep Rate (Hz)', color=color_rep, fontsize=14, fontweight='bold')
        ax3_right.plot(valid_times_rep, valid_rep,
                      linewidth=0, marker='*', color=color_rep, alpha=0.7, label=f'Rep Rate ({len(valid_rep)} pts)')
        ax3_right.tick_params(axis='y', labelcolor=color_rep, labelsize=12)
        ax3_right.set_ylim(0, 21)

        ax3.set_xlabel('Time', fontsize=14, fontweight='bold')
        print(f"  ✓ Figure 3: Charge ({len(valid_charge)} pts) + Rep Rate ({len(valid_rep)} pts)")

    # ===== FIGURE 4: Total Loss (Effi) + Life Loss + Inj. Loss (Effi) =====
    # Calculate Total Loss (Effi) using pre-filtered positive injection rate and non-zero efficiency
    valid_times_total_loss_effi = pd.Series([], dtype='datetime64[ns]')
    valid_total_loss_effi = np.array([])

    if len(effi_nonzero_values) > 0 and len(inj_rate_positive_values) > 0:
        df_matched = pd.DataFrame({
            'time_datetime': effi_nonzero_times,
            'efficiency': effi_nonzero_values
        })

        df_inj = pd.DataFrame({
            'time_datetime': inj_rate_positive_times,
            'inj_rate': inj_rate_positive_values
        })
        df_matched = pd.merge(df_matched, df_inj, on='time_datetime', how='inner')

        if len(df_matched) > 0:
            effi_frac = df_matched['efficiency'].values / 100.0
            df_matched['total_loss_effi'] = (1 - effi_frac) / effi_frac * df_matched['inj_rate'].values

            valid_loss_mask = df_matched['total_loss_effi'].notna()
            df_matched_valid = df_matched[valid_loss_mask].copy()

            valid_times_total_loss_effi = df_matched_valid['time_datetime']
            valid_total_loss_effi = df_matched_valid['total_loss_effi'].values

    # Calculate Life Loss
    valid_times_life_loss = pd.Series([], dtype='datetime64[ns]')
    valid_life_loss = np.array([])

    if 'A_BM_Current_mA' in machine_df_filtered.columns and 'A_BM_Lifetime_min' in machine_df_filtered.columns:
        mask_current_life = (machine_df_filtered['A_BM_Current_mA'].notna()) & (machine_df_filtered['A_BM_Lifetime_min'].notna())
        mask_current_life = mask_current_life & (machine_df_filtered['A_BM_Lifetime_min'] > 0)

        valid_times_life_loss = machine_df_filtered.loc[mask_current_life, 'time_datetime']
        valid_current_life = machine_df_filtered.loc[mask_current_life, 'A_BM_Current_mA']
        valid_lifetime = machine_df_filtered.loc[mask_current_life, 'A_BM_Lifetime_min']

        valid_life_loss = (valid_current_life / valid_lifetime / 60.0).values

    # Calculate Inj. Loss (Effi) = Total Loss (Effi) - Life Loss (exact timestamp matching)
    valid_times_inj_loss_effi = pd.Series([], dtype='datetime64[ns]')
    valid_inj_loss_effi = np.array([])

    if len(valid_times_total_loss_effi) > 0 and len(valid_times_life_loss) > 0:
        df_total_effi = pd.DataFrame({'time_datetime': valid_times_total_loss_effi, 'total_loss_effi': valid_total_loss_effi})
        df_life = pd.DataFrame({'time_datetime': valid_times_life_loss, 'life_loss': valid_life_loss})

        df_inj_loss_effi = pd.merge(df_total_effi, df_life, on='time_datetime', how='inner')

        if len(df_inj_loss_effi) > 0:
            df_inj_loss_effi['inj_loss_effi'] = df_inj_loss_effi['total_loss_effi'] - df_inj_loss_effi['life_loss']
            valid_times_inj_loss_effi = df_inj_loss_effi['time_datetime']
            valid_inj_loss_effi = df_inj_loss_effi['inj_loss_effi'].values

    ax4.set_ylabel('Loss Rate (mA/s)', fontsize=14, fontweight='bold')
    ax4.tick_params(axis='y', labelsize=12)
    ax4.grid(True, alpha=0.3)

    ax4.plot(valid_times_total_loss_effi, valid_total_loss_effi,
            linewidth=0, marker='o', markersize=3, color='tab:green', alpha=0.7,
            label=f'Total Loss (Effi) ({len(valid_total_loss_effi)} pts)')

    ax4.plot(valid_times_life_loss, valid_life_loss,
            linewidth=0, marker='s', markersize=3, color='tab:orange', alpha=0.7,
            label=f'Life Loss ({len(valid_life_loss)} pts)')

    if len(valid_inj_loss_effi) > 0:
        ax4.plot(valid_times_inj_loss_effi, valid_inj_loss_effi,
                linewidth=0, marker='*', markersize=3, color='tab:red', alpha=0.7,
                label=f'Inj. Loss (Effi) ({len(valid_inj_loss_effi)} pts)')

    ax4.legend(loc='upper left', fontsize=10)
    ax4.set_xlabel('Time', fontsize=14, fontweight='bold')
    ax4.set_ylim(0, 10)
    print(f"  ✓ Figure 4: Total Loss (Effi) ({len(valid_total_loss_effi)} pts) + Life Loss ({len(valid_life_loss)} pts) + Inj. Loss (Effi) ({len(valid_inj_loss_effi)} pts)")

    # ===== FIGURE 5: Beam Lifetime + Beam Current =====
    if 'A_BM_Lifetime_min' in machine_df_filtered.columns and 'A_BM_Current_mA' in machine_df_filtered.columns:
        mask_lifetime = machine_df_filtered['A_BM_Lifetime_min'].notna() & (machine_df_filtered['A_BM_Lifetime_min'] > 0)
        valid_times_lifetime = machine_df_filtered.loc[mask_lifetime, 'time_datetime']
        valid_lifetime_data = machine_df_filtered.loc[mask_lifetime, 'A_BM_Lifetime_min']

        mask_current = machine_df_filtered['A_BM_Current_mA'].notna()
        valid_times_current_fig5 = machine_df_filtered.loc[mask_current, 'time_datetime']
        valid_current_fig5 = machine_df_filtered.loc[mask_current, 'A_BM_Current_mA']

        color_lifetime = 'tab:cyan'
        ax5.set_ylabel('Beam Lifetime (min)', color=color_lifetime, fontsize=14, fontweight='bold')
        ax5.plot(valid_times_lifetime, valid_lifetime_data,
                linewidth=0, marker='o', markersize = 3, color=color_lifetime, alpha=0.7, label=f'Lifetime ({len(valid_lifetime_data)} pts)')
        ax5.tick_params(axis='y', labelcolor=color_lifetime, labelsize=12)
        ax5.grid(True, alpha=0.3)
        ax5.set_ylim(0, 200)

        ax5_right = ax5.twinx()
        color_current_fig5 = 'darkorange'
        ax5_right.set_ylabel('Beam Current (mA)', color=color_current_fig5, fontsize=14, fontweight='bold')
        ax5_right.plot(valid_times_current_fig5, valid_current_fig5,
                      linewidth=2.5, color=color_current_fig5, alpha=0.7, label=f'Current ({len(valid_current_fig5)} pts)')
        ax5_right.tick_params(axis='y', labelcolor=color_current_fig5, labelsize=12)
        ax5_right.set_ylim(0, 1700)

        ax5.set_xlabel('Time', fontsize=14, fontweight='bold')
        print(f"  ✓ Figure 5: Lifetime ({len(valid_lifetime_data)} pts) + Current ({len(valid_current_fig5)} pts)")

    # ===== FIGURE 6: Total Inj (Q) + Life Loss + Inj Rate + Inj. Loss (Q) =====
    frev = 99.39e3
    valid_times_total_inj_q = pd.Series([], dtype='datetime64[ns]')
    valid_total_inj_q = np.array([])

    if 'A_INJ_Rep_ep_Hz' in machine_df_filtered.columns and 'A_Qep_BT_end_nC' in machine_df_filtered.columns:
        mask_total_inj = (machine_df_filtered['A_INJ_Rep_ep_Hz'].notna()) & (machine_df_filtered['A_Qep_BT_end_nC'].notna())
        if mask_total_inj.any():
            df_total_inj = machine_df_filtered.loc[mask_total_inj, ['time_datetime', 'A_INJ_Rep_ep_Hz', 'A_Qep_BT_end_nC']].copy()
            df_total_inj['total_inj_q'] = df_total_inj['A_INJ_Rep_ep_Hz'].values * frev * df_total_inj['A_Qep_BT_end_nC'].values * 1E-6*2
            valid_times_total_inj_q = df_total_inj['time_datetime']
            valid_total_inj_q = df_total_inj['total_inj_q'].values

    valid_times_inj_rate_fig6 = inj_rate_positive_times
    valid_inj_rate_fig6 = inj_rate_positive_values

    valid_times_inj_loss_q = pd.Series([], dtype='datetime64[ns]')
    valid_inj_loss_q = np.array([])
    if len(valid_times_total_inj_q) > 0 and len(valid_times_life_loss) > 0 and len(valid_times_inj_rate_fig6) > 0:
        df_total_q = pd.DataFrame({'time_datetime': valid_times_total_inj_q, 'total_inj_q': valid_total_inj_q})
        df_life = pd.DataFrame({'time_datetime': valid_times_life_loss, 'life_loss': valid_life_loss})
        df_inj_rate = pd.DataFrame({'time_datetime': valid_times_inj_rate_fig6, 'inj_rate': valid_inj_rate_fig6})

        df_inj_loss_q = pd.merge(df_total_q, df_life, on='time_datetime', how='inner')
        df_inj_loss_q = pd.merge(df_inj_loss_q, df_inj_rate, on='time_datetime', how='inner')

        if len(df_inj_loss_q) > 0:
            df_inj_loss_q['inj_loss_q'] = df_inj_loss_q['total_inj_q'] - df_inj_loss_q['life_loss'] - df_inj_loss_q['inj_rate']
            df_inj_loss_q = df_inj_loss_q[df_inj_loss_q['inj_rate'] >= 0]
            valid_times_inj_loss_q = df_inj_loss_q['time_datetime']
            valid_inj_loss_q = df_inj_loss_q['inj_loss_q'].values

    ax6.set_ylabel('Rate (mA/s)', fontsize=14, fontweight='bold')
    ax6.tick_params(axis='y', labelsize=12)
    ax6.grid(True, alpha=0.3)
    ax6.plot(valid_times_total_inj_q, valid_total_inj_q,
            linewidth=0, marker='o', markersize=3, color='tab:blue', alpha=0.7,
            label=f'Total Inj (Q) ({len(valid_total_inj_q)} pts)')
    ax6.plot(valid_times_life_loss, valid_life_loss,
            linewidth=0, marker='s', markersize=3, color='tab:orange', alpha=0.7,
            label=f'Life Loss ({len(valid_life_loss)} pts)')
    ax6.plot(valid_times_inj_rate_fig6, valid_inj_rate_fig6,
            linewidth=0, marker='^', markersize=3, color='tab:green', alpha=0.7,
            label=f'Inj Rate ({len(valid_inj_rate_fig6)} pts)')
    if len(valid_inj_loss_q) > 0:
        ax6.plot(valid_times_inj_loss_q, valid_inj_loss_q,
                linewidth=0, marker='*', markersize=3, color='tab:red', alpha=0.7,
                label=f'Inj. Loss (Q) ({len(valid_inj_loss_q)} pts)')
    ax6.legend(loc='upper left', fontsize=10)
    ax6.set_xlabel('Time', fontsize=14, fontweight='bold')
    ax6.set_ylim(0, 10)
    print(f"  ✓ Figure 6: Total Inj (Q) ({len(valid_total_inj_q)} pts) + Life Loss ({len(valid_life_loss)} pts) + Inj Rate ({len(valid_inj_rate_fig6)} pts) + Inj. Loss (Q) ({len(valid_inj_loss_q)} pts)")

    # ===== FIGURE 7: Imported Efficiency vs Calculated Efficiency =====
    valid_times_effi_imported = pd.Series([], dtype='datetime64[ns]')
    valid_effi_imported = np.array([])

    if len(inj_rate_positive_values) > 0:
        mask_effi_imported = machine_df_filtered['A_INJ_Effi'].notna()
        df_all_effi_imported = pd.DataFrame({
            'time_datetime': machine_df_filtered.loc[mask_effi_imported, 'time_datetime'],
            'effi_imported': machine_df_filtered.loc[mask_effi_imported, 'A_INJ_Effi'].values
        })
        df_all_effi_imported = df_all_effi_imported[df_all_effi_imported['effi_imported'] != 0]
        df_inj_pos = pd.DataFrame({'time_datetime': inj_rate_positive_times})
        df_effi_merged = pd.merge(df_inj_pos, df_all_effi_imported, on='time_datetime', how='inner')
        if len(df_effi_merged) > 0:
            valid_times_effi_imported = df_effi_merged['time_datetime']
            valid_effi_imported = df_effi_merged['effi_imported'].values

    valid_times_effi_calc = pd.Series([], dtype='datetime64[ns]')
    valid_effi_calc = np.array([])
    if len(inj_rate_positive_values) > 0 and len(valid_times_total_inj_q) > 0:
        df_inj_rate = pd.DataFrame({'time_datetime': valid_times_inj_rate_fig6, 'inj_rate': valid_inj_rate_fig6})
        df_total = pd.DataFrame({'time_datetime': valid_times_total_inj_q, 'total_inj_q': valid_total_inj_q})
        df_effi_calc = pd.merge(df_inj_rate, df_total, on='time_datetime', how='inner')
        if len(df_effi_calc) > 0:
            df_effi_calc['effi_calc'] = (df_effi_calc['inj_rate'] / df_effi_calc['total_inj_q']) * 100.0
            valid_times_effi_calc = df_effi_calc['time_datetime']
            valid_effi_calc = df_effi_calc['effi_calc'].values

    ax7.set_ylabel('Injection efficiency (%)', fontsize=14, fontweight='bold')
    ax7.tick_params(axis='y', labelsize=12)
    ax7.grid(True, alpha=0.3)
    ax7.plot(valid_times_effi_imported, valid_effi_imported,
            linewidth=0, marker='o', markersize=3, color='tab:red', alpha=0.7,
            label=f'Imported ({len(valid_effi_imported)} pts)')
    ax7.plot(valid_times_effi_calc, valid_effi_calc,
            linewidth=0, marker='*', markersize=3, color='tab:blue', alpha=0.7,
            label=f'Calculated ({len(valid_effi_calc)} pts)')
    ax7.legend(loc='upper left', fontsize=10)
    ax7.set_xlabel('Time', fontsize=14, fontweight='bold')
    ax7.set_ylim(0, 110)
    print(f"  ✓ Figure 7: Imported Efficiency ({len(valid_effi_imported)} pts) + Calculated Efficiency ({len(valid_effi_calc)} pts)")

    # ===== FIGURE 8: Inj. Loss (Effi) vs Inj. Loss (Q) =====
    ax8.set_ylabel('Injection Loss (mA/s)', fontsize=14, fontweight='bold')
    ax8.tick_params(axis='y', labelsize=12)
    ax8.grid(True, alpha=0.3)
    ax8.plot(valid_times_inj_loss_effi, valid_inj_loss_effi,
            linewidth=0, marker='o', markersize=3, color='tab:red', alpha=0.7,
            label=f'Inj. Loss (Effi) ({len(valid_inj_loss_effi)} pts)')
    if len(valid_inj_loss_q) > 0:
        ax8.plot(valid_times_inj_loss_q, valid_inj_loss_q,
                linewidth=0, marker='s', markersize=3, color='tab:blue', alpha=0.7,
                label=f'Inj. Loss (Q) ({len(valid_inj_loss_q)} pts)')
    ax8.legend(loc='upper left', fontsize=10)
    ax8.set_xlabel('Time', fontsize=14, fontweight='bold')
    ax8.set_ylim(0, 8)
    print(f"  ✓ Figure 8: Inj. Loss (Effi) ({len(valid_inj_loss_effi)} pts) + Inj. Loss (Q) ({len(valid_inj_loss_q)} pts)")

    # Mark abort events on all subplots (only SBL and BeamLoss)
    if abort_today is not None and len(abort_today) > 0:
        sbl_beamloss_events = abort_today[abort_today['Category'].isin(['SBL', 'BeamLoss'])]
        if len(sbl_beamloss_events) > 0:
            print(f"\n  ✓ Marking {len(sbl_beamloss_events)} SBL/BeamLoss abort event(s) on all panels")
            for idx, row in sbl_beamloss_events.iterrows():
                abort_time = row['abort_datetime']
                if abort_time >= filtered_df['time_datetime'].min() and abort_time <= filtered_df['time_datetime'].max():
                    line_color = 'red'
                    text_color = 'darkred'

                    # Mark on all eight axes
                    for ax in [ax1, ax2, ax3, ax4, ax5, ax6, ax7, ax8]:
                        ax.axvline(x=abort_time, color=line_color, linestyle='--', linewidth=2, alpha=0.8, zorder=10)
                        y_pos = ax.get_ylim()[1] * 0.95
                        label_text = f"{row['Category']} {abort_time.strftime('%H:%M:%S')}"
                        ax.text(abort_time, y_pos, label_text,
                               rotation=90, verticalalignment='top', horizontalalignment='right',
                               fontsize=10, color=text_color, fontweight='bold',
                               bbox=dict(boxstyle='round,pad=0.3', facecolor='white', alpha=0.0, edgecolor=text_color))

    # Format x-axes
    for ax in [ax1, ax2, ax3, ax4, ax5, ax6, ax7, ax8]:
        ax.xaxis.set_major_formatter(mdates.DateFormatter('%H:%M'))
        ax.tick_params(axis='x', labelsize=12)
        ax.set_xlim(start_time_obj, end_time_obj)

    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.savefig(f"/Users/xylu/Desktop/Data/acoustic_vpp/daily_plots/{date_str}_machine_variables_panels.png",
               dpi=150, bbox_inches='tight')
    plt.show()

    print("\n✓ Plotted 8-panel daily summary with abort events")
else:
    print("⚠ Missing required data for plots")

In [ ]:
# AUTOMATIC BATCH PROCESSING: Generate daily plots with same filtering/calculations as Cell 3 and Cell 5
import re
import gc

print("=" * 80)
print("AUTOMATIC BATCH PROCESSING - Daily 8-Panel Machine Variables Plots")
print("=" * 80)

# Create output folders
output_plot_folder = "/Users/xylu/Desktop/Data/acoustic_vpp/daily_plots"
output_data_folder = "/Users/xylu/Desktop/Data/acoustic_vpp/daily_calculated_data"
os.makedirs(output_plot_folder, exist_ok=True)
os.makedirs(output_data_folder, exist_ok=True)
print(f"\nOutput plot folder: {output_plot_folder}")
print(f"Output data folder: {output_data_folder}")

base_path = "/Users/xylu/Desktop/Data/acoustic_vpp/"
machine_base_path = "/Users/xylu/Desktop/Data/Machine_Inj_BG_byDay"

all_folders = [f for f in os.listdir(base_path) if os.path.isdir(os.path.join(base_path, f))]
date_pattern = re.compile(r"^\d{8}$")
date_folders = sorted([f for f in all_folders if date_pattern.match(f)])

print(f"\nFound {len(date_folders)} date folders to process")

# Load abort event data once
abort_data_path = "/Users/xylu/Desktop/Data/Complete_LER_Event_Data_Analysis_Summary.xlsx"
if os.path.exists(abort_data_path):
    abort_df_all = pd.read_excel(abort_data_path)
    abort_df_all["abort_datetime"] = pd.to_datetime(abort_df_all["Time"], format="%m/%d/%y %H:%M", errors="coerce")
    abort_df_all = abort_df_all.dropna(subset=["abort_datetime"])
    print(f"✓ Loaded {len(abort_df_all)} total abort events from Excel")
else:
    print(f"⚠ Abort data file not found: {abort_data_path}")
    abort_df_all = None

print("\n" + "=" * 80)
print("Processing dates...")
print("=" * 80)


def read_acoustic_for_day(csv_files_auto, start_time_obj_auto, end_time_obj_auto):
    daily_chunks = []
    for csv_file in csv_files_auto:
        try:
            df_temp = pd.read_csv(csv_file)
            if "time_datetime" not in df_temp.columns:
                continue

            df_temp["time_datetime"] = pd.to_datetime(df_temp["time_datetime"], errors="coerce")
            df_temp = df_temp.dropna(subset=["time_datetime"])

            mask_local = (df_temp["time_datetime"] >= start_time_obj_auto) & (df_temp["time_datetime"] <= end_time_obj_auto)
            df_temp = df_temp.loc[mask_local, [c for c in ["time_datetime", "vpp_volts"] if c in df_temp.columns]]

            if len(df_temp) > 0:
                daily_chunks.append(df_temp)
        except Exception as e:
            print(f"  ✗ Error reading acoustic file {os.path.basename(csv_file)}: {e}")
            continue

    if not daily_chunks:
        return pd.DataFrame()

    out_df = pd.concat(daily_chunks, ignore_index=True)
    out_df = out_df.sort_values("time_datetime").drop_duplicates(subset=["time_datetime"], keep="last").reset_index(drop=True)
    return out_df


def read_machine_for_day(machine_csv_files_auto, start_time_obj_auto, end_time_obj_auto):
    machine_chunks = []
    for csv_file in machine_csv_files_auto:
        try:
            df_temp = pd.read_csv(csv_file)

            if "#date" in df_temp.columns:
                df_temp["#date"] = pd.to_datetime(df_temp["#date"], errors="coerce")
                df_temp.rename(columns={"#date": "time_datetime"}, inplace=True)
            elif "Timestamp" in df_temp.columns:
                df_temp["Timestamp"] = pd.to_datetime(df_temp["Timestamp"], errors="coerce")
                df_temp.rename(columns={"Timestamp": "time_datetime"}, inplace=True)
            else:
                continue

            df_temp = df_temp.dropna(subset=["time_datetime"])
            mask_local = (df_temp["time_datetime"] >= start_time_obj_auto) & (df_temp["time_datetime"] <= end_time_obj_auto)
            df_temp = df_temp.loc[mask_local]

            if len(df_temp) > 0:
                machine_chunks.append(df_temp)
        except Exception as e:
            print(f"  ✗ Error reading machine file {os.path.basename(csv_file)}: {e}")
            continue

    if not machine_chunks:
        return pd.DataFrame()

    out_df = pd.concat(machine_chunks, ignore_index=True)
    out_df = out_df.sort_values("time_datetime").drop_duplicates(subset=["time_datetime"], keep="last").reset_index(drop=True)
    out_df.rename(columns=get_friendly_name, inplace=True)
    return out_df


# Process each date
for date_idx, date_str_auto in enumerate(date_folders, start=1):
    print(f"\n[{date_str_auto}] Processing ({date_idx}/{len(date_folders)})...")

    start_time_obj_auto = datetime.strptime(date_str_auto + "000000", "%Y%m%d%H%M%S")
    end_time_obj_auto = datetime.strptime(date_str_auto + "235959", "%Y%m%d%H%M%S")

    date_folder_auto = os.path.join(base_path, date_str_auto)
    csv_files_auto = sorted(glob.glob(os.path.join(date_folder_auto, f"{date_str_auto}*.csv")))
    if len(csv_files_auto) == 0:
        print("  ⚠ No acoustic CSV files found, skipping")
        continue

    mmdd_auto = date_str_auto[4:8]
    machine_folder_auto = os.path.join(machine_base_path, mmdd_auto)
    if not os.path.exists(machine_folder_auto):
        print(f"  ⚠ Machine folder not found: {machine_folder_auto}, skipping")
        continue

    machine_csv_files_auto = sorted(glob.glob(os.path.join(machine_folder_auto, "*.csv")))
    if len(machine_csv_files_auto) == 0:
        print("  ⚠ No machine CSV files found, skipping")
        continue

    try:
        filtered_df_auto = read_acoustic_for_day(csv_files_auto, start_time_obj_auto, end_time_obj_auto)
        machine_df_filtered_auto = read_machine_for_day(machine_csv_files_auto, start_time_obj_auto, end_time_obj_auto)

        if len(filtered_df_auto) == 0 or len(machine_df_filtered_auto) == 0:
            print("  ⚠ Missing required data for plots, skipping")
            continue

        print(f"  ✓ Loaded {len(filtered_df_auto)} acoustic samples")
        print(f"  ✓ Loaded {len(machine_df_filtered_auto)} machine data samples")

        if "A_BM_Inj_Rate_mAps" in machine_df_filtered_auto.columns:
            mask_inj_positive_auto = machine_df_filtered_auto["A_BM_Inj_Rate_mAps"].notna() & (machine_df_filtered_auto["A_BM_Inj_Rate_mAps"] >= 0)
            inj_rate_positive_auto = machine_df_filtered_auto.loc[mask_inj_positive_auto, ["time_datetime", "A_BM_Inj_Rate_mAps"]].copy()
            inj_rate_positive_times_auto = inj_rate_positive_auto["time_datetime"]
            inj_rate_positive_values_auto = inj_rate_positive_auto["A_BM_Inj_Rate_mAps"].values
        else:
            inj_rate_positive_times_auto = pd.Series([], dtype="datetime64[ns]")
            inj_rate_positive_values_auto = np.array([])

        if "A_INJ_Effi" in machine_df_filtered_auto.columns:
            mask_effi_nonzero_auto = machine_df_filtered_auto["A_INJ_Effi"].notna() & (machine_df_filtered_auto["A_INJ_Effi"] != 0)
            effi_nonzero_auto = machine_df_filtered_auto.loc[mask_effi_nonzero_auto, ["time_datetime", "A_INJ_Effi"]].copy()
            effi_nonzero_times_auto = effi_nonzero_auto["time_datetime"]
            effi_nonzero_values_auto = effi_nonzero_auto["A_INJ_Effi"].values
        else:
            effi_nonzero_times_auto = pd.Series([], dtype="datetime64[ns]")
            effi_nonzero_values_auto = np.array([])

        if abort_df_all is not None:
            abort_date_start_auto = datetime.strptime(date_str_auto, "%Y%m%d").replace(hour=0, minute=0, second=0)
            abort_date_end_auto = abort_date_start_auto + timedelta(days=1)
            abort_mask_auto = (abort_df_all["abort_datetime"] >= abort_date_start_auto) & (abort_df_all["abort_datetime"] < abort_date_end_auto)
            abort_today_auto = abort_df_all.loc[abort_mask_auto].copy()
        else:
            abort_today_auto = None

        fig, (ax1, ax2, ax3, ax4, ax5, ax6, ax7, ax8) = plt.subplots(8, 1, figsize=(12, 22))
        date_formatted_auto = datetime.strptime(date_str_auto, "%Y%m%d").strftime("%Y-%m-%d")
        fig.suptitle(f"Acoustic, current, injection - {date_formatted_auto}", fontsize=12, fontweight="bold", y=0.995)

        # ===== FIGURE 1: Vpp + Beam Current =====
        if "A_BM_Current_mA" in machine_df_filtered_auto.columns and "vpp_volts" in filtered_df_auto.columns:
            mask_current_auto = machine_df_filtered_auto["A_BM_Current_mA"].notna()
            valid_times_current_auto = machine_df_filtered_auto.loc[mask_current_auto, "time_datetime"]
            valid_current_auto = machine_df_filtered_auto.loc[mask_current_auto, "A_BM_Current_mA"]

            color_vpp = "tab:blue"
            ax1.set_ylabel("Bottom Acoustic Vpp (V)", color=color_vpp, fontsize=14, fontweight="bold")
            ax1.plot(filtered_df_auto["time_datetime"], filtered_df_auto["vpp_volts"],
                     linewidth=2.5, color=color_vpp, alpha=0.7, label=f"CH3 Vpp ({len(filtered_df_auto)} pts)")
            ax1.tick_params(axis="y", labelcolor=color_vpp, labelsize=12)
            ax1.set_ylim(0, 18)
            ax1.grid(True, alpha=0.3)

            ax1_right = ax1.twinx()
            color_current = "darkorange"
            ax1_right.set_ylabel("Beam Current (mA)", color=color_current, fontsize=14, fontweight="bold")
            ax1_right.plot(valid_times_current_auto, valid_current_auto,
                           linewidth=2.5, color=color_current, alpha=0.7, label=f"Beam Current ({len(valid_current_auto)} pts)")
            ax1_right.tick_params(axis="y", labelcolor=color_current, labelsize=12)
            ax1_right.set_ylim(0, 1700)

        # ===== FIGURE 2: Injection Rate + Injection Efficiency =====
        if len(inj_rate_positive_values_auto) > 0 and len(effi_nonzero_values_auto) > 0:
            color_inj_rate = "tab:green"
            ax2.set_ylabel("Injection Rate (mA/s)", color=color_inj_rate, fontsize=14, fontweight="bold")
            ax2.plot(inj_rate_positive_times_auto, inj_rate_positive_values_auto,
                     linewidth=0, marker="o", markersize=3, color=color_inj_rate, alpha=0.7,
                     label=f"Inj Rate (positive: {len(inj_rate_positive_values_auto)} pts)")
            ax2.tick_params(axis="y", labelcolor=color_inj_rate, labelsize=12)
            ax2.set_ylim(0, 6)
            ax2.grid(True, alpha=0.3)

            ax2_right = ax2.twinx()
            color_effi = "tab:red"
            ax2_right.set_ylabel("Injection efficiency (%)", color=color_effi, fontsize=14, fontweight="bold")
            ax2_right.plot(effi_nonzero_times_auto, effi_nonzero_values_auto,
                           linewidth=0, marker="s", markersize=3, color=color_effi, alpha=0.7,
                           label=f"Efficiency (non-zero: {len(effi_nonzero_values_auto)} pts)")
            ax2_right.tick_params(axis="y", labelcolor=color_effi, labelsize=12)
            ax2_right.set_ylim(0, 110)

        # ===== FIGURE 3: Charge + Injection Rep Rate =====
        valid_times_charge_auto = pd.Series([], dtype="datetime64[ns]")
        valid_charge_auto = np.array([])
        valid_times_rep_auto = pd.Series([], dtype="datetime64[ns]")
        valid_rep_auto = np.array([])
        if "A_Qep_BT_end_nC" in machine_df_filtered_auto.columns and "A_INJ_Rep_ep_Hz" in machine_df_filtered_auto.columns:
            mask_charge_auto = machine_df_filtered_auto["A_Qep_BT_end_nC"].notna()
            valid_times_charge_auto = machine_df_filtered_auto.loc[mask_charge_auto, "time_datetime"]
            valid_charge_auto = machine_df_filtered_auto.loc[mask_charge_auto, "A_Qep_BT_end_nC"]

            mask_rep_auto = machine_df_filtered_auto["A_INJ_Rep_ep_Hz"].notna()
            valid_times_rep_auto = machine_df_filtered_auto.loc[mask_rep_auto, "time_datetime"]
            valid_rep_auto = machine_df_filtered_auto.loc[mask_rep_auto, "A_INJ_Rep_ep_Hz"]

            color_charge = "tab:brown"
            ax3.set_ylabel("Charge BT end (nC)", color=color_charge, fontsize=14, fontweight="bold")
            ax3.plot(valid_times_charge_auto, valid_charge_auto,
                     linewidth=0, marker="o", color=color_charge, alpha=0.7, label=f"Charge ({len(valid_charge_auto)} pts)")
            ax3.tick_params(axis="y", labelcolor=color_charge, labelsize=12)
            ax3.set_ylim(0, 3)
            ax3.grid(True, alpha=0.3)

            ax3_right = ax3.twinx()
            color_rep = "tab:purple"
            ax3_right.set_ylabel("Injection Rep Rate (Hz)", color=color_rep, fontsize=14, fontweight="bold")
            ax3_right.plot(valid_times_rep_auto, valid_rep_auto,
                           linewidth=0, marker="*", color=color_rep, alpha=0.7, label=f"Rep Rate ({len(valid_rep_auto)} pts)")
            ax3_right.tick_params(axis="y", labelcolor=color_rep, labelsize=12)
            ax3_right.set_ylim(0, 21)
        ax3.set_xlabel("Time", fontsize=14, fontweight="bold")

        # ===== FIGURE 4: Total Loss (Effi) + Life Loss + Inj. Loss (Effi) =====
        valid_times_total_loss_effi_auto = pd.Series([], dtype="datetime64[ns]")
        valid_total_loss_effi_auto = np.array([])
        if len(effi_nonzero_values_auto) > 0 and len(inj_rate_positive_values_auto) > 0:
            df_matched_auto = pd.DataFrame({"time_datetime": effi_nonzero_times_auto, "efficiency": effi_nonzero_values_auto})
            df_inj_auto = pd.DataFrame({"time_datetime": inj_rate_positive_times_auto, "inj_rate": inj_rate_positive_values_auto})
            df_matched_auto = pd.merge(df_matched_auto, df_inj_auto, on="time_datetime", how="inner")
            if len(df_matched_auto) > 0:
                effi_frac_auto = df_matched_auto["efficiency"].values / 100.0
                df_matched_auto["total_loss_effi"] = (1 - effi_frac_auto) / effi_frac_auto * df_matched_auto["inj_rate"].values
                df_matched_valid_auto = df_matched_auto[df_matched_auto["total_loss_effi"].notna()].copy()
                valid_times_total_loss_effi_auto = df_matched_valid_auto["time_datetime"]
                valid_total_loss_effi_auto = df_matched_valid_auto["total_loss_effi"].values

        valid_times_life_loss_auto = pd.Series([], dtype="datetime64[ns]")
        valid_life_loss_auto = np.array([])
        if "A_BM_Current_mA" in machine_df_filtered_auto.columns and "A_BM_Lifetime_min" in machine_df_filtered_auto.columns:
            mask_current_life_auto = machine_df_filtered_auto["A_BM_Current_mA"].notna() & machine_df_filtered_auto["A_BM_Lifetime_min"].notna()
            mask_current_life_auto = mask_current_life_auto & (machine_df_filtered_auto["A_BM_Lifetime_min"] > 0)
            valid_times_life_loss_auto = machine_df_filtered_auto.loc[mask_current_life_auto, "time_datetime"]
            valid_current_life_auto = machine_df_filtered_auto.loc[mask_current_life_auto, "A_BM_Current_mA"]
            valid_lifetime_auto = machine_df_filtered_auto.loc[mask_current_life_auto, "A_BM_Lifetime_min"]
            valid_life_loss_auto = (valid_current_life_auto / valid_lifetime_auto / 60.0).values

        valid_times_inj_loss_effi_auto = pd.Series([], dtype="datetime64[ns]")
        valid_inj_loss_effi_auto = np.array([])
        if len(valid_times_total_loss_effi_auto) > 0 and len(valid_times_life_loss_auto) > 0:
            df_total_effi_auto = pd.DataFrame({"time_datetime": valid_times_total_loss_effi_auto, "total_loss_effi": valid_total_loss_effi_auto})
            df_life_auto = pd.DataFrame({"time_datetime": valid_times_life_loss_auto, "life_loss": valid_life_loss_auto})
            df_inj_loss_effi_auto = pd.merge(df_total_effi_auto, df_life_auto, on="time_datetime", how="inner")
            if len(df_inj_loss_effi_auto) > 0:
                df_inj_loss_effi_auto["inj_loss_effi"] = df_inj_loss_effi_auto["total_loss_effi"] - df_inj_loss_effi_auto["life_loss"]
                valid_times_inj_loss_effi_auto = df_inj_loss_effi_auto["time_datetime"]
                valid_inj_loss_effi_auto = df_inj_loss_effi_auto["inj_loss_effi"].values

        ax4.set_ylabel("Loss Rate (mA/s)", fontsize=14, fontweight="bold")
        ax4.tick_params(axis="y", labelsize=12)
        ax4.grid(True, alpha=0.3)
        ax4.plot(valid_times_total_loss_effi_auto, valid_total_loss_effi_auto,
                 linewidth=0, marker="o", markersize=3, color="tab:green", alpha=0.7,
                 label=f"Total Loss (Effi) ({len(valid_total_loss_effi_auto)} pts)")
        ax4.plot(valid_times_life_loss_auto, valid_life_loss_auto,
                 linewidth=0, marker="s", markersize=3, color="tab:orange", alpha=0.7,
                 label=f"Life Loss ({len(valid_life_loss_auto)} pts)")
        if len(valid_inj_loss_effi_auto) > 0:
            ax4.plot(valid_times_inj_loss_effi_auto, valid_inj_loss_effi_auto,
                     linewidth=0, marker="*", markersize=3, color="tab:red", alpha=0.7,
                     label=f"Inj. Loss (Effi) ({len(valid_inj_loss_effi_auto)} pts)")
        ax4.legend(loc="upper left", fontsize=10)
        ax4.set_xlabel("Time", fontsize=14, fontweight="bold")
        ax4.set_ylim(0, 10)

        # ===== FIGURE 5: Beam Lifetime + Beam Current =====
        valid_times_lifetime_data_auto = pd.Series([], dtype="datetime64[ns]")
        valid_lifetime_data_auto = np.array([])
        valid_times_current_fig5_auto = pd.Series([], dtype="datetime64[ns]")
        valid_current_fig5_auto = np.array([])
        if "A_BM_Lifetime_min" in machine_df_filtered_auto.columns and "A_BM_Current_mA" in machine_df_filtered_auto.columns:
            mask_lifetime_auto = machine_df_filtered_auto["A_BM_Lifetime_min"].notna() & (machine_df_filtered_auto["A_BM_Lifetime_min"] > 0)
            valid_times_lifetime_data_auto = machine_df_filtered_auto.loc[mask_lifetime_auto, "time_datetime"]
            valid_lifetime_data_auto = machine_df_filtered_auto.loc[mask_lifetime_auto, "A_BM_Lifetime_min"]

            mask_current_fig5_auto = machine_df_filtered_auto["A_BM_Current_mA"].notna()
            valid_times_current_fig5_auto = machine_df_filtered_auto.loc[mask_current_fig5_auto, "time_datetime"]
            valid_current_fig5_auto = machine_df_filtered_auto.loc[mask_current_fig5_auto, "A_BM_Current_mA"]

            color_lifetime = "tab:cyan"
            ax5.set_ylabel("Beam Lifetime (min)", color=color_lifetime, fontsize=14, fontweight="bold")
            ax5.plot(valid_times_lifetime_data_auto, valid_lifetime_data_auto,
                     linewidth=0, marker="o", markersize=3, color=color_lifetime, alpha=0.7,
                     label=f"Lifetime ({len(valid_lifetime_data_auto)} pts)")
            ax5.tick_params(axis="y", labelcolor=color_lifetime, labelsize=12)
            ax5.grid(True, alpha=0.3)
            ax5.set_ylim(0, 200)

            ax5_right = ax5.twinx()
            color_current_fig5 = "darkorange"
            ax5_right.set_ylabel("Beam Current (mA)", color=color_current_fig5, fontsize=14, fontweight="bold")
            ax5_right.plot(valid_times_current_fig5_auto, valid_current_fig5_auto,
                           linewidth=2.5, color=color_current_fig5, alpha=0.7,
                           label=f"Current ({len(valid_current_fig5_auto)} pts)")
            ax5_right.tick_params(axis="y", labelcolor=color_current_fig5, labelsize=12)
            ax5_right.set_ylim(0, 1700)
        ax5.set_xlabel("Time", fontsize=14, fontweight="bold")

        # ===== FIGURE 6: Total Inj (Q) + Life Loss + Inj Rate + Inj. Loss (Q) =====
        frev = 99.39e3
        bunch_number = 1 if start_time_obj_auto < pd.Timestamp('2026-01-30 20:00:00') else 2
        valid_times_total_inj_q_auto = pd.Series([], dtype="datetime64[ns]")
        valid_total_inj_q_auto = np.array([])
        if "A_INJ_Rep_ep_Hz" in machine_df_filtered_auto.columns and "A_Qep_BT_end_nC" in machine_df_filtered_auto.columns:
            mask_total_inj_auto = machine_df_filtered_auto["A_INJ_Rep_ep_Hz"].notna() & machine_df_filtered_auto["A_Qep_BT_end_nC"].notna()
            if mask_total_inj_auto.any():
                df_total_inj_auto = machine_df_filtered_auto.loc[mask_total_inj_auto, ["time_datetime", "A_INJ_Rep_ep_Hz", "A_Qep_BT_end_nC"]].copy()
                df_total_inj_auto["total_inj_q"] = df_total_inj_auto["A_INJ_Rep_ep_Hz"].values * frev * df_total_inj_auto["A_Qep_BT_end_nC"].values * 1e-6 * bunch_number
                valid_times_total_inj_q_auto = df_total_inj_auto["time_datetime"]
                valid_total_inj_q_auto = df_total_inj_auto["total_inj_q"].values

        valid_times_inj_rate_fig6_auto = inj_rate_positive_times_auto
        valid_inj_rate_fig6_auto = inj_rate_positive_values_auto

        valid_times_inj_loss_q_auto = pd.Series([], dtype="datetime64[ns]")
        valid_inj_loss_q_auto = np.array([])
        if len(valid_times_total_inj_q_auto) > 0 and len(valid_times_life_loss_auto) > 0 and len(valid_times_inj_rate_fig6_auto) > 0:
            df_total_q_auto = pd.DataFrame({"time_datetime": valid_times_total_inj_q_auto, "total_inj_q": valid_total_inj_q_auto})
            df_life_auto = pd.DataFrame({"time_datetime": valid_times_life_loss_auto, "life_loss": valid_life_loss_auto})
            df_inj_rate_auto = pd.DataFrame({"time_datetime": valid_times_inj_rate_fig6_auto, "inj_rate": valid_inj_rate_fig6_auto})
            df_inj_loss_q_auto = pd.merge(df_total_q_auto, df_life_auto, on="time_datetime", how="inner")
            df_inj_loss_q_auto = pd.merge(df_inj_loss_q_auto, df_inj_rate_auto, on="time_datetime", how="inner")
            if len(df_inj_loss_q_auto) > 0:
                df_inj_loss_q_auto["inj_loss_q"] = df_inj_loss_q_auto["total_inj_q"] - df_inj_loss_q_auto["life_loss"] - df_inj_loss_q_auto["inj_rate"]
                df_inj_loss_q_auto = df_inj_loss_q_auto[df_inj_loss_q_auto["inj_rate"] >= 0]
                valid_times_inj_loss_q_auto = df_inj_loss_q_auto["time_datetime"]
                valid_inj_loss_q_auto = df_inj_loss_q_auto["inj_loss_q"].values

        ax6.set_ylabel("Rate (mA/s)", fontsize=14, fontweight="bold")
        ax6.tick_params(axis="y", labelsize=12)
        ax6.grid(True, alpha=0.3)
        ax6.plot(valid_times_total_inj_q_auto, valid_total_inj_q_auto,
                 linewidth=0, marker="o", markersize=3, color="tab:blue", alpha=0.7,
                 label=f"Total Inj (Q) ({len(valid_total_inj_q_auto)} pts)")
        ax6.plot(valid_times_life_loss_auto, valid_life_loss_auto,
                 linewidth=0, marker="s", markersize=3, color="tab:orange", alpha=0.7,
                 label=f"Life Loss ({len(valid_life_loss_auto)} pts)")
        ax6.plot(valid_times_inj_rate_fig6_auto, valid_inj_rate_fig6_auto,
                 linewidth=0, marker="^", markersize=3, color="tab:green", alpha=0.7,
                 label=f"Inj Rate ({len(valid_inj_rate_fig6_auto)} pts)")
        if len(valid_inj_loss_q_auto) > 0:
            ax6.plot(valid_times_inj_loss_q_auto, valid_inj_loss_q_auto,
                     linewidth=0, marker="*", markersize=3, color="tab:red", alpha=0.7,
                     label=f"Inj. Loss (Q) ({len(valid_inj_loss_q_auto)} pts)")
        ax6.legend(loc="upper left", fontsize=10)
        ax6.set_xlabel("Time", fontsize=14, fontweight="bold")
        ax6.set_ylim(0, 10)

        # ===== FIGURE 7: Imported Efficiency vs Calculated Efficiency =====
        valid_times_effi_imported_auto = pd.Series([], dtype="datetime64[ns]")
        valid_effi_imported_auto = np.array([])
        if len(inj_rate_positive_values_auto) > 0 and "A_INJ_Effi" in machine_df_filtered_auto.columns:
            mask_effi_imported_auto = machine_df_filtered_auto["A_INJ_Effi"].notna()
            df_all_effi_imported_auto = pd.DataFrame({
                "time_datetime": machine_df_filtered_auto.loc[mask_effi_imported_auto, "time_datetime"],
                "effi_imported": machine_df_filtered_auto.loc[mask_effi_imported_auto, "A_INJ_Effi"].values,
            })
            df_all_effi_imported_auto = df_all_effi_imported_auto[df_all_effi_imported_auto["effi_imported"] != 0]
            df_inj_pos_auto = pd.DataFrame({"time_datetime": inj_rate_positive_times_auto})
            df_effi_merged_auto = pd.merge(df_inj_pos_auto, df_all_effi_imported_auto, on="time_datetime", how="inner")
            if len(df_effi_merged_auto) > 0:
                valid_times_effi_imported_auto = df_effi_merged_auto["time_datetime"]
                valid_effi_imported_auto = df_effi_merged_auto["effi_imported"].values

        valid_times_effi_calc_auto = pd.Series([], dtype="datetime64[ns]")
        valid_effi_calc_auto = np.array([])
        if len(inj_rate_positive_values_auto) > 0 and len(valid_times_total_inj_q_auto) > 0:
            df_inj_rate_auto = pd.DataFrame({"time_datetime": valid_times_inj_rate_fig6_auto, "inj_rate": valid_inj_rate_fig6_auto})
            df_total_auto = pd.DataFrame({"time_datetime": valid_times_total_inj_q_auto, "total_inj_q": valid_total_inj_q_auto})
            df_effi_calc_auto = pd.merge(df_inj_rate_auto, df_total_auto, on="time_datetime", how="inner")
            if len(df_effi_calc_auto) > 0:
                df_effi_calc_auto["effi_calc"] = (df_effi_calc_auto["inj_rate"] / df_effi_calc_auto["total_inj_q"]) * 100.0
                valid_times_effi_calc_auto = df_effi_calc_auto["time_datetime"]
                valid_effi_calc_auto = df_effi_calc_auto["effi_calc"].values

        ax7.set_ylabel("Injection efficiency (%)", fontsize=14, fontweight="bold")
        ax7.tick_params(axis="y", labelsize=12)
        ax7.grid(True, alpha=0.3)
        ax7.plot(valid_times_effi_imported_auto, valid_effi_imported_auto,
                 linewidth=0, marker="o", markersize=3, color="tab:red", alpha=0.7,
                 label=f"Imported ({len(valid_effi_imported_auto)} pts)")
        ax7.plot(valid_times_effi_calc_auto, valid_effi_calc_auto,
                 linewidth=0, marker="*", markersize=3, color="tab:blue", alpha=0.7,
                 label=f"Calculated ({len(valid_effi_calc_auto)} pts)")
        ax7.legend(loc="upper left", fontsize=10)
        ax7.set_xlabel("Time", fontsize=14, fontweight="bold")
        ax7.set_ylim(0, 110)

        # ===== FIGURE 8: Inj. Loss (Effi) vs Inj. Loss (Q) =====
        ax8.set_ylabel("Injection Loss (mA/s)", fontsize=14, fontweight="bold")
        ax8.tick_params(axis="y", labelsize=12)
        ax8.grid(True, alpha=0.3)
        ax8.plot(valid_times_inj_loss_effi_auto, valid_inj_loss_effi_auto,
                 linewidth=0, marker="o", markersize=3, color="tab:red", alpha=0.7,
                 label=f"Inj. Loss (Effi) ({len(valid_inj_loss_effi_auto)} pts)")
        if len(valid_inj_loss_q_auto) > 0:
            ax8.plot(valid_times_inj_loss_q_auto, valid_inj_loss_q_auto,
                     linewidth=0, marker="s", markersize=3, color="tab:blue", alpha=0.7,
                     label=f"Inj. Loss (Q) ({len(valid_inj_loss_q_auto)} pts)")
        ax8.legend(loc="upper left", fontsize=10)
        ax8.set_xlabel("Time", fontsize=14, fontweight="bold")
        ax8.set_ylim(0, 8)

        if abort_today_auto is not None and len(abort_today_auto) > 0:
            sbl_beamloss_events_auto = abort_today_auto[abort_today_auto["Category"].isin(["SBL", "BeamLoss"])]
            if len(sbl_beamloss_events_auto) > 0 and len(filtered_df_auto) > 0:
                day_min = filtered_df_auto["time_datetime"].min()
                day_max = filtered_df_auto["time_datetime"].max()
                for _, row in sbl_beamloss_events_auto.iterrows():
                    abort_time = row["abort_datetime"]
                    if day_min <= abort_time <= day_max:
                        for ax in [ax1, ax2, ax3, ax4, ax5, ax6, ax7, ax8]:
                            ax.axvline(x=abort_time, color="red", linestyle="--", linewidth=2, alpha=0.8, zorder=10)
                            y_pos = ax.get_ylim()[1] * 0.95
                            label_text = f"{row['Category']} {abort_time.strftime('%H:%M:%S')}"
                            ax.text(abort_time, y_pos, label_text,
                                    rotation=90, verticalalignment="top", horizontalalignment="right",
                                    fontsize=10, color="darkred", fontweight="bold",
                                    bbox=dict(boxstyle="round,pad=0.3", facecolor="white", alpha=0.0, edgecolor="darkred"))

        for ax in [ax1, ax2, ax3, ax4, ax5, ax6, ax7, ax8]:
            ax.xaxis.set_major_formatter(mdates.DateFormatter("%H:%M"))
            ax.tick_params(axis="x", labelsize=12)
            ax.set_xlim(start_time_obj_auto, end_time_obj_auto)

        plt.xticks(rotation=45)
        plt.tight_layout()

        plot_filename = f"{date_str_auto}_machine_variables_panels.png"
        plot_filepath = os.path.join(output_plot_folder, plot_filename)
        plt.savefig(plot_filepath, dpi=150, bbox_inches="tight")
        plt.close(fig)

        # Export calculated series for this date
        df_export_auto = pd.DataFrame({"time_datetime": machine_df_filtered_auto["time_datetime"]})
        if len(valid_times_total_loss_effi_auto) > 0:
            df_export_auto = pd.merge(df_export_auto, pd.DataFrame({"time_datetime": valid_times_total_loss_effi_auto, "total_loss_effi_mAps": valid_total_loss_effi_auto}), on="time_datetime", how="left")
        if len(valid_times_life_loss_auto) > 0:
            df_export_auto = pd.merge(df_export_auto, pd.DataFrame({"time_datetime": valid_times_life_loss_auto, "life_loss_mAps": valid_life_loss_auto}), on="time_datetime", how="left")
        if len(valid_times_inj_loss_effi_auto) > 0:
            df_export_auto = pd.merge(df_export_auto, pd.DataFrame({"time_datetime": valid_times_inj_loss_effi_auto, "inj_loss_effi_mAps": valid_inj_loss_effi_auto}), on="time_datetime", how="left")
        if len(valid_times_total_inj_q_auto) > 0:
            df_export_auto = pd.merge(df_export_auto, pd.DataFrame({"time_datetime": valid_times_total_inj_q_auto, "total_inj_q_mAps": valid_total_inj_q_auto}), on="time_datetime", how="left")
        if len(valid_times_inj_rate_fig6_auto) > 0:
            df_export_auto = pd.merge(df_export_auto, pd.DataFrame({"time_datetime": valid_times_inj_rate_fig6_auto, "inj_rate_positive_mAps": valid_inj_rate_fig6_auto}), on="time_datetime", how="left")
        if len(valid_times_inj_loss_q_auto) > 0:
            df_export_auto = pd.merge(df_export_auto, pd.DataFrame({"time_datetime": valid_times_inj_loss_q_auto, "inj_loss_q_mAps": valid_inj_loss_q_auto}), on="time_datetime", how="left")
        if len(valid_times_effi_imported_auto) > 0:
            df_export_auto = pd.merge(df_export_auto, pd.DataFrame({"time_datetime": valid_times_effi_imported_auto, "effi_imported_pct": valid_effi_imported_auto}), on="time_datetime", how="left")
        if len(valid_times_effi_calc_auto) > 0:
            df_export_auto = pd.merge(df_export_auto, pd.DataFrame({"time_datetime": valid_times_effi_calc_auto, "effi_calc_pct": valid_effi_calc_auto}), on="time_datetime", how="left")
        df_export_auto = df_export_auto.drop_duplicates(subset=["time_datetime"]).sort_values("time_datetime").reset_index(drop=True)

        export_filename = f"{date_str_auto}_calculated_series.csv"
        export_filepath = os.path.join(output_data_folder, export_filename)
        df_export_auto.to_csv(export_filepath, index=False)

        print(f"  ✓ Saved plot: {plot_filename}")
        print(f"  ✓ Saved data: {export_filename}")

    except Exception as e:
        print(f"  ✗ Error processing {date_str_auto}: {e}")

    finally:
        gc.collect()

print("\n" + "=" * 80)
print("BATCH PROCESSING COMPLETE!")
print(f"All plots saved to: {output_plot_folder}")
print(f"All calculated data saved to: {output_data_folder}")
print("=" * 80)

In [ ]:
# CELL 1: Import libraries, load all data, and generate merged arrays

# Self-contained setup so this cell runs without the batch-processing cell above.
import os
import re
import glob
import gc
import pandas as pd
import numpy as np
from matplotlib.colors import TwoSlopeNorm

print("="*80)
print("CORRELATION ANALYSIS - STEP 1: Data Loading and Merging")
print("="*80)

# Paths (self-contained defaults)
base_path_corr = "/Users/xylu/Desktop/Data/acoustic_vpp/"
machine_base_path_corr = "/Users/xylu/Desktop/Data/Machine_Inj_BG_byDay"

# Rebuild date list locally
all_folders_corr = [f for f in os.listdir(base_path_corr) if os.path.isdir(os.path.join(base_path_corr, f))]
date_pattern_corr = re.compile(r'^\d{8}$')
date_folders = sorted([f for f in all_folders_corr if date_pattern_corr.match(f)])

print(f"\nFound {len(date_folders)} date folders in {base_path_corr}")

# Build a local PV mapping for this cell every run (do not depend on globals).
if 'pv_mapping' in globals():
    pv_dict_local = {k: v for k, v in pv_mapping}
else:
    # Fallback mapping needed for correlation cells
    pv_dict_local = {
        'BMLDCCT:CURRENT': 'A_BM_Current_mA',
        'BMLDCCT:RATE': 'A_BM_Inj_Rate_mAps',
        'CGLINJ:EFFICIENCY': 'A_INJ_Effi',
        'LIiEV:BEAM_REP:READ:KBP': 'A_INJ_Rep_ep_Hz',
        'BTpBPM:QMD11P_K_1:NC_1Hz:C': 'A_Qep_BT_end_nC',
        'BMLDCCT:LIFE': 'A_BM_Lifetime_min',
        'B2_nsm:get:ECL_LUM_MON:lum_acc_20': 'A_LUMI_30',
        'CG_OPR:SpecificLuminosity': 'A_LUMI_SP_30',
        "RFLAE:D06:OSC1:CH1:VPP": "ACOU_Top_Vpp",
        "RFLAE:D06:OSC1:CH2:VPP": "ACOU_BmLoss_Vpp",
        "RFLAE:D06:OSC1:CH3:VPP": "ACOU_Btm_Vpp"
    }

# Keep only columns needed downstream to avoid memory spikes
required_columns = [
    'time_datetime',
    'vpp_volts',
    'A_BM_Current_mA',
    'A_BM_Inj_Rate_mAps',
    'A_INJ_Effi',
    'A_INJ_Rep_ep_Hz',
    'A_Qep_BT_end_nC',
    'A_BM_Lifetime_min',
    'A_LUMI_30',
    'A_LUMI_SP_30',
    "ACOU_Top_Vpp",
    "ACOU_BmLoss_Vpp",
    "ACOU_Btm_Vpp"
]

# Store merged data for each date
merged_data_by_date = {}

print(f"\nLoading and merging data from {len(date_folders)} dates...\n")

for idx, date_str_corr in enumerate(date_folders, start=1):
    date_folder_corr = os.path.join(base_path_corr, date_str_corr)
    csv_files_corr = sorted(glob.glob(os.path.join(date_folder_corr, f"{date_str_corr}*.csv")))

    if len(csv_files_corr) == 0:
        continue

    # Load acoustic data for this date
    all_data_corr = []
    for csv_file in csv_files_corr:
        try:
            df_temp = pd.read_csv(csv_file)
            if 'time_datetime' in df_temp.columns:
                df_temp['time_datetime'] = pd.to_datetime(df_temp['time_datetime'], errors='coerce')
                df_temp = df_temp.dropna(subset=['time_datetime'])
                all_data_corr.append(df_temp)
        except Exception:
            continue

    if not all_data_corr:
        continue

    acoustic_df_corr = pd.concat(all_data_corr, ignore_index=True)
    acoustic_df_corr = acoustic_df_corr.sort_values('time_datetime').drop_duplicates(subset=['time_datetime'], keep='last')

    mmdd_corr = date_str_corr[4:8]
    machine_folder_corr = os.path.join(machine_base_path_corr, mmdd_corr)
    if not os.path.exists(machine_folder_corr):
        del all_data_corr, acoustic_df_corr
        gc.collect()
        continue

    machine_csv_files_corr = sorted(glob.glob(os.path.join(machine_folder_corr, "*.csv")))
    if len(machine_csv_files_corr) == 0:
        del all_data_corr, acoustic_df_corr
        gc.collect()
        continue

    machine_data_list_corr = []
    for csv_file in machine_csv_files_corr:
        try:
            df_temp = pd.read_csv(csv_file)
            if '#date' in df_temp.columns:
                df_temp['#date'] = pd.to_datetime(df_temp['#date'], errors='coerce')
                df_temp.rename(columns={'#date': 'time_datetime'}, inplace=True)
            elif 'Timestamp' in df_temp.columns:
                df_temp['Timestamp'] = pd.to_datetime(df_temp['Timestamp'], errors='coerce')
                df_temp.rename(columns={'Timestamp': 'time_datetime'}, inplace=True)
            else:
                continue

            df_temp = df_temp.dropna(subset=['time_datetime'])
            machine_data_list_corr.append(df_temp)
        except Exception:
            continue

    if not machine_data_list_corr:
        del all_data_corr, acoustic_df_corr
        gc.collect()
        continue

    machine_df_corr = pd.concat(machine_data_list_corr, ignore_index=True)
    machine_df_corr = machine_df_corr.sort_values('time_datetime').drop_duplicates(subset=['time_datetime'], keep='last')
    machine_df_corr.columns = machine_df_corr.columns.str.strip()
    machine_df_corr.rename(columns=lambda c: pv_dict_local.get(c, c), inplace=True)

    # PRE-FILTER: Keep only positive injection rate AND non-zero efficiency
    if 'A_BM_Inj_Rate_mAps' in machine_df_corr.columns and 'A_INJ_Effi' in machine_df_corr.columns:
        rows_before = len(machine_df_corr)
        pre_mask = (
            (machine_df_corr['A_BM_Inj_Rate_mAps'] > 0)
            & (machine_df_corr['A_INJ_Effi'] != 0)
            & machine_df_corr['A_BM_Inj_Rate_mAps'].notna()
            & machine_df_corr['A_INJ_Effi'].notna()
        )
        machine_df_corr = machine_df_corr.loc[pre_mask].copy()
        rows_after = len(machine_df_corr)
        print(f"    [Machine filter] Removed {rows_before - rows_after} invalid rows ({rows_after} kept with positive inj_rate & non-zero efficiency)")

    merged_df = pd.merge_asof(
        acoustic_df_corr.sort_values('time_datetime'),
        machine_df_corr.sort_values('time_datetime'),
        on='time_datetime',
        direction='nearest',
        tolerance=pd.Timedelta('2s')
    )

    keep_cols = [c for c in required_columns if c in merged_df.columns]
    merged_df = merged_df[keep_cols].copy()

    # Calculate derived parameters for correlation
    frev = 99.39e3
    bunch_number = 1 if int(date_str_corr) < 20260130 else 2

    # Life loss = current / lifetime / 60 (per second)
    merged_df['life_loss'] = np.where(
        merged_df['A_BM_Current_mA'].notna() & merged_df['A_BM_Lifetime_min'].notna() & (merged_df['A_BM_Lifetime_min'] > 0),
        merged_df['A_BM_Current_mA'] / merged_df['A_BM_Lifetime_min'] / 60.0,
        np.nan
    )

    # Total inj. rate (Q) = rep_rate * frev * charge * 1e-6 * bunch_number
    merged_df['total_inj_q'] = np.where(
        merged_df['A_INJ_Rep_ep_Hz'].notna() & merged_df['A_Qep_BT_end_nC'].notna(),
        merged_df['A_INJ_Rep_ep_Hz'] * frev * merged_df['A_Qep_BT_end_nC'] * 1e-6 * bunch_number,
        np.nan
    )

    # Inj. loss (Q) = total_inj_q - life_loss - inj_rate
    merged_df['inj_loss_q'] = np.where(
        merged_df['total_inj_q'].notna() & merged_df['life_loss'].notna() & merged_df['A_BM_Inj_Rate_mAps'].notna(),
        merged_df['total_inj_q'] - merged_df['life_loss'] - merged_df['A_BM_Inj_Rate_mAps'],
        np.nan
    )

    # Parameters for correlation
    params_for_correlation = [
        'A_BM_Current_mA',
        'A_BM_Inj_Rate_mAps',
        'A_INJ_Effi',
        'A_INJ_Rep_ep_Hz',
        'total_inj_q',
        'inj_loss_q',
        'A_BM_Lifetime_min',
        'life_loss',
        'A_LUMI_30',
        'A_LUMI_SP_30',
        "ACOU_Top_Vpp",
        "ACOU_BmLoss_Vpp",
        "ACOU_Btm_Vpp"
        
    ]

    # For compatibility with downstream cells
    parameters_to_analyze = params_for_correlation

    # Ensure all correlation columns exist before counting/filtering.
    for param in params_for_correlation:
        if param not in merged_df.columns:
            merged_df[param] = np.nan

    print(f"[{date_str_corr}] Parameter counts before filtering:")
    for param in params_for_correlation:
        count = merged_df[param].notna().sum()
        print(f"  {param}: {count} points")

    # Keep only rows where all parameters are present
    merged_df = merged_df.dropna(subset=params_for_correlation).copy()

    # Keep only necessary columns
    keep_cols_final = ['time_datetime', 'vpp_volts'] + params_for_correlation
    merged_df = merged_df[keep_cols_final].copy()

    # Downcast numeric columns where possible to reduce memory footprint
    for col in merged_df.columns:
        if col != 'time_datetime' and pd.api.types.is_float_dtype(merged_df[col]):
            merged_df[col] = pd.to_numeric(merged_df[col], downcast='float')

    merged_data_by_date[date_str_corr] = merged_df
    print(f"[{date_str_corr}] After filtering: {len(merged_df):,} data points with all parameters present")

    del all_data_corr, machine_data_list_corr, acoustic_df_corr, machine_df_corr, merged_df
    if idx % 3 == 0:
        gc.collect()

print("\nCombining data from all dates...")
if len(merged_data_by_date) > 0:
    all_merged_df = pd.concat(merged_data_by_date.values(), ignore_index=True)
    print(f"Total combined data points: {len(all_merged_df):,}")
    print(f"\n✓ Data loading complete! Stored {len(merged_data_by_date)} dates.")
else:
    all_merged_df = pd.DataFrame()
    print("⚠ No merged data was created. Check source folders and file formats.")

In [ ]:
# CELL: Daily Summary Dataset - Abort Frequency vs Machine Data with Actual Abort Counts

print("\n" + "="*80)
print("BUILDING DAILY SUMMARY DATASET - Abort Frequency vs Machine Data")
print("="*80)

import subprocess
import sys
from datetime import datetime, timedelta

# Ensure openpyxl is installed
subprocess.check_call([sys.executable, "-m", "pip", "install", "openpyxl", "-q"])

# Load abort data first
abort_data_path = "/Users/xylu/Desktop/Data/Complete_LER_Event_Data_Analysis_Summary.xlsx"
abort_df = None

if os.path.exists(abort_data_path):
    abort_df = pd.read_excel(abort_data_path)
    abort_df['abort_datetime'] = pd.to_datetime(abort_df['Time'], format='%m/%d/%y %H:%M')
    print(f"✓ Loaded {len(abort_df)} total abort events")
    print(f"  Date range: {abort_df['abort_datetime'].min()} to {abort_df['abort_datetime'].max()}")
else:
    print(f"⚠ Abort data file not found: {abort_data_path}")

# Initialize summary dataset
daily_summary = []

# Define high current threshold
HIGH_CURRENT_THRESHOLD = 500  # mA

# Get list of dates
dates_to_analyze = sorted(merged_data_by_date.keys())

for date_str in dates_to_analyze:
    merged_df = merged_data_by_date[date_str]
    
    if len(merged_df) == 0:
        continue
    
    # --- HIGH CURRENT OPERATION (>= 500 mA) ---
    high_current_mask = (merged_df['A_BM_Current_mA'] >= HIGH_CURRENT_THRESHOLD) & merged_df['A_BM_Current_mA'].notna()
    
    if high_current_mask.sum() == 0:
        continue  # Skip if no high current data
    
    high_current_df = merged_df.loc[high_current_mask].copy()
    
    # --- ABORT COUNTS ---
    sbl_count = 0
    beamloss_count = 0
    
    if abort_df is not None:
        # Create date range for this day
        abort_date_start = pd.to_datetime(date_str, format='%Y%m%d')
        abort_date_end = abort_date_start + timedelta(days=1)
        
        # Filter abort events for this date
        abort_mask = (abort_df['abort_datetime'] >= abort_date_start) & (abort_df['abort_datetime'] < abort_date_end)
        aborts_today = abort_df[abort_mask]
        
        if len(aborts_today) > 0:
            # Count SBL aborts
            sbl_mask = aborts_today['Category'].str.contains('SBL', case=False, na=False)
            sbl_count = sbl_mask.sum()
            
            # Count BeamLoss aborts
            beam_mask = aborts_today['Category'].str.contains('BeamLoss|Beam Loss|BML', case=False, na=False)
            beamloss_count = beam_mask.sum()
    
    # Calculate averages during high current operation
    summary_row = {
        'date': date_str,
        'n_sbl_aborts': sbl_count,
        'n_beamloss_aborts': beamloss_count,
        'total_aborts': sbl_count + beamloss_count,
        'n_hc_samples': high_current_mask.sum(),  # Number of samples at high current
    }
    
    # Acoustic signal averages
    if 'vpp_volts' in high_current_df.columns:
        summary_row['acoustic_vpp_avg'] = high_current_df['vpp_volts'].mean()
        summary_row['acoustic_vpp_std'] = high_current_df['vpp_volts'].std()
    else:
        summary_row['acoustic_vpp_avg'] = np.nan
        summary_row['acoustic_vpp_std'] = np.nan
    
    # Other acoustic channels
    for acou_param in ['ACOU_Top_Vpp', 'ACOU_BmLoss_Vpp', 'ACOU_Btm_Vpp']:
        if acou_param in high_current_df.columns:
            summary_row[f'{acou_param}_avg'] = high_current_df[acou_param].mean()
            summary_row[f'{acou_param}_std'] = high_current_df[acou_param].std()
        else:
            summary_row[f'{acou_param}_avg'] = np.nan
            summary_row[f'{acou_param}_std'] = np.nan
    
    # Loss rates averages
    loss_metrics = ['inj_loss_q', 'total_inj_q', 'life_loss']
    for metric in loss_metrics:
        if metric in high_current_df.columns:
            summary_row[f'{metric}_avg'] = high_current_df[metric].mean()
            summary_row[f'{metric}_std'] = high_current_df[metric].std()
        else:
            summary_row[f'{metric}_avg'] = np.nan
            summary_row[f'{metric}_std'] = np.nan
    
    # Machine parameters during high current
    machine_metrics = ['A_BM_Inj_Rate_mAps', 'A_INJ_Effi', 'A_BM_Lifetime_min', 'A_LUMI_30', 'A_LUMI_SP_30']
    for metric in machine_metrics:
        if metric in high_current_df.columns:
            summary_row[f'{metric}_avg'] = high_current_df[metric].mean()
            summary_row[f'{metric}_std'] = high_current_df[metric].std()
        else:
            summary_row[f'{metric}_avg'] = np.nan
            summary_row[f'{metric}_std'] = np.nan
    
    # Current stats
    summary_row['current_avg'] = high_current_df['A_BM_Current_mA'].mean()
    summary_row['current_max'] = high_current_df['A_BM_Current_mA'].max()
    
    daily_summary.append(summary_row)

# Create summary dataframe
daily_summary_df = pd.DataFrame(daily_summary)

print(f"\n✓ Created daily summary dataset with {len(daily_summary_df)} days")
print(f"✓ Columns included: {len(daily_summary_df.columns)} metrics")
print(f"\nDate range: {daily_summary_df['date'].min()} to {daily_summary_df['date'].max()}")
print(f"\nDataset shape: {daily_summary_df.shape}")

print(f"\n✓ Abort Summary:")
print(f"  Total SBL Aborts: {daily_summary_df['n_sbl_aborts'].sum():.0f}")
print(f"  Total BeamLoss Aborts: {daily_summary_df['n_beamloss_aborts'].sum():.0f}")
print(f"  Total Aborts: {daily_summary_df['total_aborts'].sum():.0f}")

print(f"\nDays with Aborts:")
days_with_aborts = daily_summary_df[daily_summary_df['total_aborts'] > 0][['date', 'n_sbl_aborts', 'n_beamloss_aborts', 'total_aborts']]
if len(days_with_aborts) > 0:
    print(days_with_aborts.to_string(index=False))
else:
    print("  No aborts recorded during high current operation")

print(f"\n✓ First few rows:")
print(daily_summary_df.head())
print(f"\n{'='*80}")